# Auditing components.yaml

This notebook runs the AuditSystem against the main components.yaml file to evaluate solution quality.

In [ ]:
from pathlib import Path

import ibis as ib
from ibis import _

from iacs.architect import Architect

ib.options.interactive = True  # auto-display results (like pandas)

## Load manifest

Loads both the builtins YAML (component type definitions) and the iacs Python source tree (modules, classes, and functions with docstrings or `__iacs__` metadata).

In [ ]:
import atexit

BUILTINS_DIR = Path.cwd().parent / "builtins"
IACS_SRC_DIR = Path.cwd().parent / "iacs"

a = Architect.from_manifest([BUILTINS_DIR, IACS_SRC_DIR])
atexit.register(a.registry.close)

## Evaluate highest-priority tasks to work on

In [ ]:
entity_ids = a.get("entity_id")
entity_ids

In [ ]:
parents = a.get("parent")
parents

In [ ]:
reqs = a.get("requirement")
reqs

In [ ]:
priority_product = a.get("priority_product")

# Highest priority leaf requirements (nodes with no children)
(
    reqs
    .join(parents, reqs.entity_id == parents.parent_id, how="anti")
    .join(entity_ids.select(_.value, _.entity_key, _.path), _.entity_id == entity_ids.value)
    .join(priority_product, "entity_id")
    .order_by(ib.desc(_.priority_product))
    .select(_.entity_key, _.priority_product, _.value.name("requirement_type"))
)

In [ ]:
a.get("solution")

In [ ]:
a.load_dataflow("etl.derive_components")

In [ ]:
a.execute("field_types_with_entity_ref", validated_registry=a.registry)